### Data Integration Steps

Below our documented process for our we integrated the spotify and million song datasets 

In [1]:
import pandas as pd 
import recordlinkage as rl
df_spotify = pd.read_csv("../data/processed/SpotifyCleaned.csv", index_col=0)
df_million_song = pd.read_csv("../data/processed/MillionSongCleaned.csv", index_col=0)

#### Exact Join

We began by performing an exact join between the two datasets. We performed this exact join based on the song's artist(s) and title. Specifically, we used the lowercase verison of the fields that was created during the data cleaning process, ArtistLower and SongLower.

In [2]:
df_exact_join = pd.merge(
    df_spotify,
    df_million_song,
    left_on=['ArtistLower', 'SongLower'],  
    right_on=['ArtistLower', 'SongLower'], 
    how='inner'                        
)
print("Number of exact matches:", len(df_exact_join))

Number of exact matches: 99


Next, we removed the observations that had exact matches from their respective datasets so that all that is remaining is the unmatched observations from each dataset.

In [3]:
spotify_idxs = df_exact_join.set_index(['ArtistLower', 'SongLower']).index
df_spotify_remaining = df_spotify[~df_spotify.set_index(['ArtistLower', 'SongLower']).index.isin(spotify_idxs)].copy()

million_idxs = df_exact_join.set_index(['ArtistLower', 'SongLower']).index
df_million_song_remaining = df_million_song[~df_million_song.set_index(['ArtistLower', 'SongLower']).index.isin(million_idxs)].copy()

#### Approximate Matching

In addition to exact matching, we also performed approximate matching based on the song's artist(s) and title, and using the ArtistLowr and SongLower fields. To accomplish this approximate matching, we first blocked our data based on the first word of the ArtistLower fields, reducing the number of comparisons that would need to be make. Then, we matched on a combination of the song title and artist, using levenshtein distance for both and thresholds of 0.5 and 0.6 respectively. Given that this was producing a limited number of matched, we intentionally used slighly lower than optimal thresholds with the intention of manually reviewing all matches.

In [4]:
def first_word(name):
    first = name.strip().split()[0].lower()
    if first == "the" and len(name.strip().split()) > 1:
        first = name.strip().split()[1].lower()
    return first

df_spotify_remaining["FirstWord"] = df_spotify_remaining["ArtistLower"].apply(first_word)
df_million_song_remaining["FirstWord"] = df_million_song_remaining["ArtistLower"].apply(first_word)


indexer = rl.Index()
indexer.block("FirstWord")
pairs = indexer.index(df_spotify_remaining, df_million_song_remaining)


def get_matched_pairs(candidates, left_df, right_df):
    match_pairs = candidates.index.to_frame(index=False)
    match_pairs.columns = ["left_index", "right_index"]

    merged = (
        match_pairs
        .merge(left_df, left_on="left_index", right_index=True, suffixes=('', '_left'))
        .merge(right_df, left_on="right_index", right_index=True, suffixes=('_left', '_right'))
    )
    return merged


compare = rl.Compare()
compare.string("ArtistLower", "ArtistLower", method="levenshtein", threshold=0.5, label="Artist_Sim")
compare.string("SongLower", "SongLower", method="levenshtein", threshold=0.6, label="Song_Sim")

features = compare.compute(pairs, df_spotify_remaining, df_million_song_remaining)

candidates = features[
    (features["Artist_Sim"] == 1) &
    (features["Song_Sim"] == 1)
]

df_approx_matches = get_matched_pairs(candidates, df_spotify_remaining, df_million_song_remaining)
print("Initial Number of approximate matches:", len(df_approx_matches))

Initial Number of approximate matches: 49


After gathering the initial approximate matches, we reviewed each candidate match and removed all incorrect matches. 

In [5]:
pd.set_option('display.max_rows', None)
df_approx_matches[['ArtistLower_right', 'ArtistLower_left', 'SongLower_right', 'SongLower_left']]

,ArtistLower_right,ArtistLower_left,SongLower_right,SongLower_left
0,eddie turner,eddie vedder,rise,rise
1,eddie turner,eddie neblett,rise,river
2,behemoth,behemoth,chant for eschaton 2000,chant for ezkaton 2000 e. v.
3,winds of plague,winds of plague,soldiers of doomsday,soldiers of doomsday
4,chris rea,chris rea,driving home for christmas,driving home for christmas - 2019 remaster
5,orbital,orbital,are we here ?,are we here?
6,dub pistols feat. tk & ashley slater,dub pistols;ashley slater,everyday stranger,everyday stranger
7,orbital,orbital,chime (edit),chime - edit
8,harry gregson-williams,harry gregson-williams,the ball,the battle
9,sean paul,sean paul;beyoncé,baby boy [feat. beyonce],baby boy (feat. beyoncé )


After removing the incorrect matches, we reviewed the matches one more time to ensure accuracy. At this point, we had sucessfully obtained our approxmate match dataframe.

In [6]:
df_approx_matches = df_approx_matches.drop(index=[1,8,12,25,26,28,29,31,34,36,38,39,43,44,48])
df_approx_matches[['ArtistLower_right', 'ArtistLower_left', 'SongLower_right', 'SongLower_left']]
print("Final number of approximate matches:", len(df_approx_matches))

Final number of approximate matches: 34


In [7]:
df_exact_join.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key_x', 'loudness', 'mode_x', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre', 'ArtistLower', 'SongLower', 'SongNumber', 'SongID',
       'AlbumID', 'AlbumName', 'ArtistID', 'ArtistName', 'Duration',
       'KeySignature', 'KeySignatureConfidence', 'Tempo', 'TimeSignature',
       'TimeSignatureConfidence', 'Title', 'mbID', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'key_y', 'keyConfidence', 'Loudness',
       'mode_y', 'mode_confidence', 'start_of_fade_out'],
      dtype='object')

In [8]:
df_approx_matches.columns

Index(['left_index', 'right_index', 'Unnamed: 0', 'track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key_left', 'loudness', 'mode_left',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre', 'ArtistLower_left',
       'SongLower_left', 'FirstWord_left', 'SongNumber', 'SongID', 'AlbumID',
       'AlbumName', 'ArtistID', 'ArtistName', 'Duration', 'KeySignature',
       'KeySignatureConfidence', 'Tempo', 'TimeSignature',
       'TimeSignatureConfidence', 'Title', 'mbID', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'key_right', 'keyConfidence', 'Loudness',
       'mode_right', 'mode_confidence', 'start_of_fade_out',
       'ArtistLower_right', 'SongLower_right', 'FirstWord_right'],
      dtype='object')

#### Integrated Schema Selection
Next, we determined our desired schema for our integrated dataset.In the event of fields that appeared in both datasets, we decided to use the value from the spotify dataset. This is because of research question relates to popularity, a metric in the spotify dataset, so we believe the spotify dataset's values will be more accurate in relation to the song's popularity. The result of this schema integration was that we included all fields from the spotify dataset and some of the fields from the million song dataset. Below are the specific fields from each:

Spotify Dataset: track_id, artists, album_name, track_name, popularity, duration_ms, explicit, danceability, energy, key_x, loudness, mode_x, speechiness, acousticness, instrumentalness, liveness, valence, tempo, track_genre
 
Million Song Dataset: ArtistFamiliarity, Hotness, end_of_fade_in, start_of_fade_out

In [9]:
df_exact_join = df_exact_join[['track_id','artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key_x', 'loudness', 'mode_x', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'track_genre', 
       'ArtistFamiliarity','Hotness', 'end_of_fade_in', 'start_of_fade_out']]

df_approx_matches = df_approx_matches[['track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key_left', 'loudness', 'mode_left',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'track_genre', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in',
        'start_of_fade_out']]

#### Combining Exact Join and Approximate Matches

For the final step, we standarized column names between the exact joins and approximate matches combined them with on another, which resulted in our fully integrated dataset.

In [10]:
df_exact_join.columns = ['track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'track_genre', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in','start_of_fade_out']

df_approx_matches.columns = ['track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'track_genre', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'start_of_fade_out']

df_integreated = pd.concat([df_exact_join, df_approx_matches], ignore_index=True)
df_integreated.to_csv("../data/processed/Integrated.csv")
print("Total number of observations in integrated dataset:", len(df_integreated))


Total number of observations in integrated dataset: 133
